# Benchmark Inferensi — Teacher vs Student Baseline vs Student KD

Mengukur **biaya inferensi** (menjawab Reviewer 2, poin 5): parameter, GFLOPs, ukuran model, VRAM, dan latensi / FPS di Kaggle GPU T4. Tidak ada training dan tidak ada evaluasi mAP di notebook ini.

## Persiapan
- **Settings**: Accelerator **GPU T4** (kode hanya memakai satu GPU, `cuda:0`) dan Internet ON (untuk `pip install`). Jangan ganti ke GPU lain di tengah pengukuran — latensi bergantung GPU.
- **Add Input** (tiga pertama = *Notebook Output*, dua terakhir = *Dataset*):

| Input | File yang dicari otomatis |
|---|---|
| `output-notebooks-notebook1-2-resume-teacher-ft` | `best_teacher_finetuned_9class.pt` |
| `output-notebooks-notebook2-2-resume-student-baseli` | `best_student_baseline_9class.pt` |
| `output-notebooks-notebook3-2-resume-kd` — **satu versi saja** (Run 1 = run 22 Jul 2026) | `best_student_kd_9class.pt` |
| `BDD100K` + `bdd100k_cache` | gambar test siang/malam untuk latensi end-to-end (opsional) |

Tiap file harus ada **tepat satu** di Input; notebook berhenti dengan pesan jelas kalau ada duplikat atau tidak ada. Kalau BDD100K/cache tidak di-add, bagian end-to-end dilewati dan hanya latensi forward yang diukur.

## Yang diukur
| Metrik | Cara |
|---|---|
| Parameter, GFLOPs @640 | `sum(p.numel())`, `ultralytics...get_flops` (model 9 kelas, tanpa fuse kecuali `FUSE = True`) |
| Ukuran model | ukuran file `state_dict` (FP32 dan FP16), MiB |
| VRAM | `torch.cuda.max_memory_allocated` saat inferensi (steady-state, setelah warm-up) + memori bobot |
| Latensi forward | batch 1, 640×640, warm-up 50, 300 iterasi × 3 ronde (urutan model diputar tiap ronde), `perf_counter` + `cuda.synchronize` |
| Latensi end-to-end | forward + NMS (conf 0.25, IoU 0.7) pada 100 gambar test siang dan 100 malam (tanpa decode/letterbox gambar) |

Arsitektur dibangun dari yaml 9 kelas lalu `load_state_dict(strict=True)`, jadi checkpoint yang tidak cocok akan error, bukan diam-diam salah. Adapter distilasi KD (`feat_kd_state`) tidak ikut dimuat — memang tidak dipakai saat inferensi.

## Output (`/kaggle/working`)
`inference_static.csv`, `inference_vram.csv`, `inference_latency_summary.csv` (mean ± SD antar-ronde), `inference_latency_rounds.csv` (per ronde), `inference_env.json` (GPU, driver, versi library, config, hash checkpoint). Tabel markdown dicetak di cell terakhir.

## Catatan untuk paper
- T4 **bukan perangkat edge**; ini proksi. Tetap sebutkan keterbatasan ini dan perlunya uji di edge device sungguhan.
- Student Baseline dan Student KD berarsitektur identik → params/GFLOPs/ukuran/VRAM sama persis; selisih latensi hanya noise (bandingkan dengan SD antar-ronde).
- Latensi di Kaggle bisa berfluktuasi (GPU bersama). Ulangi seluruh notebook beberapa kali bila perlu dan laporkan SD.
- VRAM = puncak alokasi PyTorch; **tidak** termasuk konteks CUDA (sekitar 300–400 MiB).


In [1]:
!pip install ultralytics==8.1.47


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.4/750.4 kB 7.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ultralytics]


In [2]:
# =============== KONFIGURASI ===============
NUM_CLASSES = 9
IMG_SIZE    = 640
REQUIRE_GPU = True            # False hanya untuk uji kode di CPU -- angka latensi CPU JANGAN dilaporkan

# file checkpoint (dicari otomatis di Input), key state_dict di dalamnya, dan arsitekturnya
MODELS = {
    "Teacher YOLOv8-L":          {"file": "best_teacher_finetuned_9class.pt", "key": "teacher_state", "arch": "yolov8l.yaml"},
    "Student Baseline YOLOv8-S": {"file": "best_student_baseline_9class.pt",  "key": "student_state", "arch": "yolov8s.yaml"},
    "Student KD YOLOv8-S":       {"file": "best_student_kd_9class.pt",        "key": "student_state", "arch": "yolov8s.yaml"},
}
CKPT_SEARCH_DIRS = ["/kaggle/input"]

PRECISIONS  = ("fp32", "fp16")   # fp16 = model.half()
BATCH_SIZES = (1,)               # 1 = skenario real-time satu kamera
N_WARMUP    = 50
N_ITER      = 300                # iterasi terukur per ronde
N_ROUNDS    = 3                  # ronde berulang; urutan model diputar tiap ronde -> SD antar-ronde
FUSE        = False              # True = lipat Conv+BN sebelum diukur (skenario deployment)

CONF_THRES, IOU_THRES  = 0.25, 0.7         # NMS untuk latensi end-to-end
N_IMAGES_PER_CONDITION = 100               # gambar test siang & malam untuk latensi end-to-end
SEED     = 0
BDD_BASE = "/kaggle/input/datasets/jimmyxwang/bdd100k"
OUT_DIR  = "/kaggle/working"
# ===========================================

import copy
import gc
import hashlib
import json
import os
import pickle
import platform
import random
import subprocess
import tempfile
import time

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import ops
from ultralytics.utils.torch_utils import get_flops

HAS_CUDA = torch.cuda.is_available()
if REQUIRE_GPU:
    assert HAS_CUDA, "GPU tidak aktif -- nyalakan Accelerator GPU T4 di Settings (latensi CPU tidak bermakna)."
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
if not HAS_CUDA:
    PRECISIONS = tuple(p for p in PRECISIONS if p == "fp32")
    print("[WARNING] Mode CPU: hanya fp32 dan VRAM dilewati. Hasil hanya untuk uji kode.")
torch.backends.cudnn.benchmark = True
os.makedirs(OUT_DIR, exist_ok=True)

GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "cpu"
print(f"[INFO] device={DEVICE} | gpu={GPU_NAME} | torch={torch.__version__} | cuda={torch.version.cuda} | ultralytics={ultralytics.__version__}")


def find_all(filename, dirs=None):
    hits = []
    for base in (dirs or CKPT_SEARCH_DIRS):
        if os.path.isdir(base):
            for root, _dirs, files in os.walk(base):
                if filename in files:
                    hits.append(os.path.join(root, filename))
    return hits


def find_unique(filename):
    hits = find_all(filename)
    assert len(hits) == 1, (f"{filename}: ditemukan {len(hits)}x di Input {hits}. "
                            "Harus tepat satu -- add hanya satu notebook-output (satu versi) yang memuat file ini.")
    return hits[0]


def _sync():
    if HAS_CUDA:
        torch.cuda.synchronize()


def nvidia_smi():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,clocks.sm,clocks.max.sm,temperature.gpu,power.draw",
             "--format=csv,noheader"], text=True, timeout=10).strip()
    except Exception:
        return ""


SMI_START = nvidia_smi()
print("[INFO] nvidia-smi:", SMI_START or "(tidak tersedia)")


[INFO] device=cuda | gpu=Tesla T4 | torch=2.11.0+cu128 | cuda=12.8 | ultralytics=8.1.47
[INFO] nvidia-smi: Tesla T4, 580.178.04, 300 MHz, 1590 MHz, 46, 10.76 W
Tesla T4, 580.178.04, 300 MHz, 1590 MHz, 54, 11.05 W


In [3]:
def build_model(arch, key, ckpt_path):
    model = DetectionModel(arch, ch=3, nc=NUM_CLASSES, verbose=False)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)   # checkpoint sendiri (dipercaya)
    meta = {k: (float(ckpt[k]) if ckpt.get(k) is not None else None)
            for k in ("epoch", "val_loss", "val_map50", "val_map50_95")}
    model.load_state_dict(ckpt[key], strict=True)      # strict: arsitektur harus persis sama dengan saat training
    del ckpt
    return model.eval().requires_grad_(False), meta


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def state_dict_mib(m, half):
    sd = {k: (v.half() if (half and v.is_floating_point()) else v) for k, v in m.state_dict().items()}
    with tempfile.NamedTemporaryFile(suffix=".pt", dir=OUT_DIR, delete=False) as f:
        path = f.name
    torch.save(sd, path)
    size = os.path.getsize(path) / 2**20
    os.remove(path)
    return size


def sha12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


T_NAME, B_NAME, K_NAME = MODELS            # urutan di MODELS: teacher, baseline, KD
models_cpu, static_rows = {}, []
for name, cfg in MODELS.items():
    path = find_unique(cfg["file"])
    m, meta = build_model(cfg["arch"], cfg["key"], path)
    models_cpu[name] = m
    row = {"model": name, "arch": cfg["arch"], "params": n_params(m), "gflops": get_flops(m, IMG_SIZE),
           "size_fp32_mib": state_dict_mib(m, False), "size_fp16_mib": state_dict_mib(m, True),
           "ckpt_file": path, "ckpt_sha256_12": sha12(path), **{f"ckpt_{k}": v for k, v in meta.items()}}
    if FUSE:
        fm = copy.deepcopy(m).fuse(verbose=False)
        row.update(params_fused=n_params(fm), gflops_fused=get_flops(fm, IMG_SIZE))
    static_rows.append(row)
    print(f"[INFO] {name:28s} params={row['params'] / 1e6:7.3f}M  GFLOPs={row['gflops']:7.2f}  "
          f"size fp32/fp16={row['size_fp32_mib']:.1f}/{row['size_fp16_mib']:.1f} MiB  "
          f"ckpt epoch={meta['epoch']} val_mAP50={meta['val_map50']}  sha={row['ckpt_sha256_12']}")
    print(f"       <- {path}")

same_arch = n_params(models_cpu[B_NAME]) == n_params(models_cpu[K_NAME])
print(f"\n[CHECK] Arsitektur Baseline == KD (jumlah parameter sama): {same_arch}")

deploy = {n: (copy.deepcopy(m).fuse(verbose=False) if FUSE else m) for n, m in models_cpu.items()}


Overriding model.yaml nc=80 with nc=9
[INFO] Teacher YOLOv8-L             params= 43.637M  GFLOPs= 165.44  size fp32/fp16=166.8/83.5 MiB  ckpt epoch=23.0 val_mAP50=0.5481  sha=d65077e0d5f1
       <- /kaggle/input/notebooks/yosephoktavianus/output-notebooks-notebook1-2-resume-teacher-ft/kd_output/best_teacher_finetuned_9class.pt
Overriding model.yaml nc=80 with nc=9
[INFO] Student Baseline YOLOv8-S    params= 11.139M  GFLOPs=  28.66  size fp32/fp16=42.7/21.4 MiB  ckpt epoch=24.0 val_mAP50=0.49005  sha=babe22bc80cb
       <- /kaggle/input/notebooks/yosephoktavianus/output-notebooks-notebook2-2-resume-student-baseli/kd_output/best_student_baseline_9class.pt
Overriding model.yaml nc=80 with nc=9
[INFO] Student KD YOLOv8-S          params= 11.139M  GFLOPs=  28.66  size fp32/fp16=42.7/21.4 MiB  ckpt epoch=25.0 val_mAP50=0.495  sha=65ab7b946b01
       <- /kaggle/input/notebooks/yosephoktavianus/output-notebooks-notebook3-2-resume-kd/kd_output/best_student_kd_9class.pt

[CHECK] Arsitektur Base

In [4]:
def letterbox(img_bgr, size=IMG_SIZE):
    h, w = img_bgr.shape[:2]
    r = size / max(h, w)
    nh, nw = round(h * r), round(w * r)
    img = cv2.resize(img_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    out = np.full((size, size, 3), 114, np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    out[top:top + nh, left:left + nw] = img
    return out


def to_tensor(img_bgr):          # uint8 HxWx3 BGR -> float32 [1,3,H,W] RGB 0..1
    return torch.from_numpy(np.ascontiguousarray(img_bgr[:, :, ::-1])).permute(2, 0, 1).unsqueeze(0).float().div(255)


real = {}                        # kondisi -> list tensor [1,3,S,S] (CPU)
try:
    cache_hits = find_all("bdd100k_cache.pkl")
    assert cache_hits, "bdd100k_cache.pkl tidak ditemukan di Input"
    with open(cache_hits[0], "rb") as f:
        cache = pickle.load(f)
    rng = random.Random(SEED)
    for cond in ("daytime", "night"):
        items = cache["test"][cond]
        picked = rng.sample(items, min(N_IMAGES_PER_CONDITION, len(items)))
        tensors = []
        for jf, data in picked:
            img_name = data.get("name", os.path.basename(jf).replace(".json", ""))
            im = cv2.imread(os.path.join(BDD_BASE, "test", "images", img_name + ".jpg"))
            if im is not None:
                tensors.append(to_tensor(letterbox(im)))
        real[cond] = tensors
        print(f"[INFO] {cond}: {len(tensors)} gambar test dimuat untuk latensi end-to-end")
    assert all(len(v) > 0 for v in real.values()), "tidak ada gambar yang berhasil dibaca"
    del cache
except Exception as e:
    real = {}
    print(f"[WARNING] Gambar BDD100K tidak bisa dimuat ({e!r}) -> latensi end-to-end DILEWATI "
          "(hanya latensi forward dengan tensor acak).")


[INFO] daytime: 100 gambar test dimuat untuk latensi end-to-end
[INFO] night: 100 gambar test dimuat untuk latensi end-to-end


In [5]:
def dtype_of(prec):
    return torch.float16 if prec == "fp16" else torch.float32


def make_model(name, prec):
    m = copy.deepcopy(deploy[name]).to(DEVICE)
    return m.half() if prec == "fp16" else m


vram_rows = []
if HAS_CUDA:
    for name in MODELS:
        for prec in PRECISIONS:
            for bs in BATCH_SIZES:
                gc.collect(); torch.cuda.empty_cache(); _sync()
                base = torch.cuda.memory_allocated()
                m = make_model(name, prec)
                weights = torch.cuda.memory_allocated() - base
                x = torch.rand(bs, 3, IMG_SIZE, IMG_SIZE, device=DEVICE, dtype=dtype_of(prec))
                with torch.no_grad():
                    for _ in range(10):                    # warm-up (cudnn autotune)
                        out = m(x)
                    _sync()
                    torch.cuda.reset_peak_memory_stats()   # ukur puncak steady-state
                    for _ in range(10):
                        out = m(x)
                    _sync()
                peak = torch.cuda.max_memory_allocated()
                vram_rows.append({"model": name, "precision": prec, "batch": bs,
                                  "weights_mib": weights / 2**20, "peak_mib": peak / 2**20})
                print(f"[INFO] VRAM {name:28s} {prec} b={bs}: bobot={weights / 2**20:7.1f} MiB | puncak={peak / 2**20:7.1f} MiB")
                del m, x, out
    gc.collect(); torch.cuda.empty_cache()
else:
    print("[WARNING] Tidak ada GPU -> pengukuran VRAM dilewati.")


[INFO] VRAM Teacher YOLOv8-L             fp32 b=1: bobot=  167.8 MiB | puncak=  277.3 MiB
[INFO] VRAM Teacher YOLOv8-L             fp16 b=1: bobot=   86.7 MiB | puncak=  141.2 MiB
[INFO] VRAM Student Baseline YOLOv8-S    fp32 b=1: bobot=   42.6 MiB | puncak=   88.2 MiB
[INFO] VRAM Student Baseline YOLOv8-S    fp16 b=1: bobot=   21.4 MiB | puncak=   44.1 MiB
[INFO] VRAM Student KD YOLOv8-S          fp32 b=1: bobot=   42.6 MiB | puncak=   88.2 MiB
[INFO] VRAM Student KD YOLOv8-S          fp16 b=1: bobot=   21.4 MiB | puncak=   44.1 MiB


In [6]:
def preds_of(out):
    return out[0] if isinstance(out, (tuple, list)) else out


def summarize(ts, bs):
    return {"mean_ms": float(ts.mean()), "median_ms": float(np.median(ts)),
            "p95_ms": float(np.percentile(ts, 95)), "fps": float(1000 * bs / ts.mean())}


@torch.no_grad()
def bench_forward(m, x):
    for _ in range(N_WARMUP):
        m(x)
    ts = []
    for _ in range(N_ITER):
        _sync(); t0 = time.perf_counter()
        m(x)
        _sync(); ts.append((time.perf_counter() - t0) * 1e3)
    return np.array(ts)


@torch.no_grad()
def bench_e2e(m, imgs, prec):
    """forward + NMS per gambar (batch 1). Pra-proses (decode/letterbox) tidak dihitung."""
    xs = [im.to(DEVICE, dtype=dtype_of(prec)) for im in imgs]
    for x in xs[:10]:                                  # warm-up
        ops.non_max_suppression(preds_of(m(x)).float(), CONF_THRES, IOU_THRES, max_det=300)
    ts, n_det = [], 0
    for x in xs:
        _sync(); t0 = time.perf_counter()
        dets = ops.non_max_suppression(preds_of(m(x)).float(), CONF_THRES, IOU_THRES, max_det=300)
        _sync(); ts.append((time.perf_counter() - t0) * 1e3)
        n_det += len(dets[0])
    return np.array(ts), n_det / len(xs)


GPU = {(n, p): make_model(n, p) for n in MODELS for p in PRECISIONS}
names = list(MODELS)
lat_rows = []
for r in range(N_ROUNDS):
    k = r % len(names)
    for name in names[k:] + names[:k]:                 # urutan diputar tiap ronde (kurangi bias urutan/termal)
        for prec in PRECISIONS:
            m = GPU[(name, prec)]
            for bs in BATCH_SIZES:
                x = torch.rand(bs, 3, IMG_SIZE, IMG_SIZE, device=DEVICE, dtype=dtype_of(prec))
                lat_rows.append({"round": r, "model": name, "precision": prec, "batch": bs, "mode": "forward",
                                 **summarize(bench_forward(m, x), bs)})
            for cond, imgs in real.items():
                ts, n_det = bench_e2e(m, imgs, prec)
                lat_rows.append({"round": r, "model": name, "precision": prec, "batch": 1, "mode": f"e2e_{cond}",
                                 "avg_dets": n_det, **summarize(ts, 1)})
    print(f"[INFO] ronde {r + 1}/{N_ROUNDS} selesai | nvidia-smi: {nvidia_smi() or '-'}")
del GPU
gc.collect()
if HAS_CUDA:
    torch.cuda.empty_cache()


[INFO] ronde 1/3 selesai | nvidia-smi: Tesla T4, 580.178.04, 1590 MHz, 1590 MHz, 71, 66.34 W
Tesla T4, 580.178.04, 300 MHz, 1590 MHz, 46, 9.96 W
[INFO] ronde 2/3 selesai | nvidia-smi: Tesla T4, 580.178.04, 1290 MHz, 1590 MHz, 78, 56.07 W
Tesla T4, 580.178.04, 300 MHz, 1590 MHz, 42, 10.46 W
[INFO] ronde 3/3 selesai | nvidia-smi: Tesla T4, 580.178.04, 1575 MHz, 1590 MHz, 78, 68.64 W
Tesla T4, 580.178.04, 300 MHz, 1590 MHz, 40, 9.57 W


In [7]:
df_static = pd.DataFrame(static_rows)
df_vram = pd.DataFrame(vram_rows)
df_runs = pd.DataFrame(lat_rows)
if "avg_dets" not in df_runs:
    df_runs["avg_dets"] = np.nan

sd = lambda s: s.std(ddof=1) if len(s) > 1 else float("nan")
df_lat = (df_runs.groupby(["model", "precision", "batch", "mode"], sort=False)
          .agg(mean_ms=("mean_ms", "mean"), sd_ms=("mean_ms", sd), median_ms=("median_ms", "mean"),
               p95_ms=("p95_ms", "mean"), fps=("fps", "mean"), avg_dets=("avg_dets", "mean"))
          .reset_index())

df_static.to_csv(f"{OUT_DIR}/inference_static.csv", index=False)
df_vram.to_csv(f"{OUT_DIR}/inference_vram.csv", index=False)
df_lat.to_csv(f"{OUT_DIR}/inference_latency_summary.csv", index=False)
df_runs.to_csv(f"{OUT_DIR}/inference_latency_rounds.csv", index=False)
env = {
    "utc_time": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "device": str(DEVICE), "gpu_name": GPU_NAME, "gpu_count": torch.cuda.device_count(),
    "nvidia_smi_start": SMI_START, "nvidia_smi_end": nvidia_smi(),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if HAS_CUDA else None,
    "ultralytics": ultralytics.__version__, "python": platform.python_version(),
    "config": {"img_size": IMG_SIZE, "precisions": list(PRECISIONS), "batch_sizes": list(BATCH_SIZES),
               "n_warmup": N_WARMUP, "n_iter": N_ITER, "n_rounds": N_ROUNDS, "fuse": FUSE,
               "conf_thres": CONF_THRES, "iou_thres": IOU_THRES, "n_images_per_condition": N_IMAGES_PER_CONDITION},
    "checkpoints": {r["model"]: {"file": r["ckpt_file"], "sha256_12": r["ckpt_sha256_12"], "epoch": r["ckpt_epoch"],
                                 "val_map50": r["ckpt_val_map50"]} for r in static_rows},
}
with open(f"{OUT_DIR}/inference_env.json", "w") as f:
    json.dump(env, f, indent=2)
print(f"[INFO] Tersimpan di {OUT_DIR}: inference_static / _vram / _latency_summary / _latency_rounds .csv + inference_env.json\n")


def pm(mean, sdv, nd=2):
    return f"{mean:.{nd}f} ± {sdv:.{nd}f}" if sdv == sdv else f"{mean:.{nd}f}"


def md(headers, rows):
    print("| " + " | ".join(headers) + " |")
    print("|" + "---|" * len(headers))
    for r in rows:
        print("| " + " | ".join(r) + " |")
    print()


print("**Tabel 1 — Ukuran model (9 kelas, 640×640)**\n")
md(["model", "parameter (M)", "GFLOPs", "ukuran FP32 (MiB)", "ukuran FP16 (MiB)", "epoch ckpt", "val mAP50 ckpt"],
   [[r.model, f"{r.params / 1e6:.2f}", f"{r.gflops:.1f}", f"{r.size_fp32_mib:.1f}", f"{r.size_fp16_mib:.1f}",
     "-" if pd.isna(r.ckpt_epoch) else f"{r.ckpt_epoch:.0f}",
     "-" if pd.isna(r.ckpt_val_map50) else f"{r.ckpt_val_map50:.4f}"] for r in df_static.itertuples()])

lat = {(r.model, r.precision, r.batch, r.mode): r for r in df_lat.itertuples()}
vram = {(r.model, r.precision, r.batch): r for r in df_vram.itertuples()}
rows = []
for name in MODELS:
    for prec in PRECISIONS:
        for bs in BATCH_SIZES:
            f = lat[(name, prec, bs, "forward")]
            v = vram.get((name, prec, bs))
            e_d, e_n = lat.get((name, prec, 1, "e2e_daytime")), lat.get((name, prec, 1, "e2e_night"))
            rows.append([name, prec.upper(), str(bs), pm(f.mean_ms, f.sd_ms), f"{f.median_ms:.2f}", f"{f.p95_ms:.2f}",
                         f"{f.fps:.1f}",
                         pm(e_d.mean_ms, e_d.sd_ms) if (e_d is not None and bs == BATCH_SIZES[0]) else "-",
                         pm(e_n.mean_ms, e_n.sd_ms) if (e_n is not None and bs == BATCH_SIZES[0]) else "-",
                         f"{v.weights_mib:.0f}" if v is not None else "-", f"{v.peak_mib:.0f}" if v is not None else "-"])
print("**Tabel 2 — Latensi & VRAM (ms, mean ± SD antar-ronde; e2e = forward + NMS)**\n")
md(["model", "presisi", "batch", "forward ms", "median", "p95", "FPS", "e2e siang ms", "e2e malam ms",
    "VRAM bobot MiB", "VRAM puncak MiB"], rows)

bs0 = BATCH_SIZES[0]
st = df_static.set_index("model")
print("**Tabel 3 — Perbandingan relatif**\n")
rel = [["parameter Teacher / Student", f"{st.loc[T_NAME, 'params'] / st.loc[B_NAME, 'params']:.2f}×"],
       ["GFLOPs Teacher / Student", f"{st.loc[T_NAME, 'gflops'] / st.loc[B_NAME, 'gflops']:.2f}×"],
       ["arsitektur Baseline == KD", str(same_arch)]]
for prec in PRECISIONS:
    t, b, k = (lat[(n, prec, bs0, "forward")] for n in (T_NAME, B_NAME, K_NAME))
    rel.append([f"percepatan forward Teacher → Student KD ({prec.upper()}, batch {bs0})", f"{t.mean_ms / k.mean_ms:.2f}×"])
    rel.append([f"selisih forward KD vs Baseline ({prec.upper()}, batch {bs0})",
                f"{(k.mean_ms - b.mean_ms) / b.mean_ms * 100:+.2f}% (SD antar-ronde: baseline {b.sd_ms:.2f} ms, KD {k.sd_ms:.2f} ms)"])
md(["perbandingan", "nilai"], rel)
print("> Latensi diukur di GPU Kaggle (bukan perangkat edge). Selisih KD vs Baseline pada arsitektur identik "
      "hanya noise bila lebih kecil dari SD antar-ronde. VRAM = puncak alokasi PyTorch (tanpa konteks CUDA).")


[INFO] Tersimpan di /kaggle/working: inference_static / _vram / _latency_summary / _latency_rounds .csv + inference_env.json

**Tabel 1 — Ukuran model (9 kelas, 640×640)**

| model | parameter (M) | GFLOPs | ukuran FP32 (MiB) | ukuran FP16 (MiB) | epoch ckpt | val mAP50 ckpt |
|---|---|---|---|---|---|---|
| Teacher YOLOv8-L | 43.64 | 165.4 | 166.8 | 83.5 | 23 | 0.5481 |
| Student Baseline YOLOv8-S | 11.14 | 28.7 | 42.7 | 21.4 | 24 | 0.4900 |
| Student KD YOLOv8-S | 11.14 | 28.7 | 42.7 | 21.4 | 25 | 0.4950 |

**Tabel 2 — Latensi & VRAM (ms, mean ± SD antar-ronde; e2e = forward + NMS)**

| model | presisi | batch | forward ms | median | p95 | FPS | e2e siang ms | e2e malam ms | VRAM bobot MiB | VRAM puncak MiB |
|---|---|---|---|---|---|---|---|---|---|---|
| Teacher YOLOv8-L | FP32 | 1 | 42.74 ± 2.74 | 42.70 | 43.52 | 23.5 | 43.46 ± 2.33 | 43.37 ± 2.58 | 168 | 277 |
| Teacher YOLOv8-L | FP16 | 1 | 17.50 ± 0.86 | 17.44 | 17.88 | 57.2 | 17.89 ± 0.88 | 17.89 ± 0.83 | 87 | 141 |
| Student 